# 04 · Stretch: join two sources (NeoWs + USGS earthquakes)

**NASA Space Apps Cairo 2026 · Bootcamp · API & Data Integration**

**Goal:** join asteroid data (NASA NeoWs) with earthquake data (USGS) **by date**, then explain an asteroid's energy in familiar terms (*"about as much energy as a magnitude-X earthquake"*).

**USGS needs no key** · **Traps:** USGS time is in *milliseconds* · GeoJSON coordinates are `[lon, lat, depth]` · one-to-many joins multiply rows.

> ✍️ **Starter notebook.** Cells marked `TODO` are yours to finish. Green ✅ check cells tell you when a step works.

Open in Colab: *File → Save a copy in Drive* first, so your edits are kept.

## Setup

In [ ]:
# ▶ Run this cell first
import os, json, requests
import pandas as pd
from datetime import date, timedelta

API_KEY = "DEMO_KEY"   # 👉 replace with YOUR free key from https://api.nasa.gov (DEMO_KEY runs out fast in a full room)
# Never paste a paid key (for example an AI key) into a notebook you share.

def check(test, good, hint):
    """Tiny self-check: prints ✅ when the step works, ❌ plus a hint when it does not."""
    try:
        ok = bool(test())
    except Exception:
        ok = False
    print(("✅ " + good) if ok else ("❌ " + hint))

def show_url(url, params):
    """Print a request URL with your key hidden, so it is safe to show on a projector."""
    req = requests.Request("GET", url, params=params).prepare()
    key = str(params.get("api_key", ""))
    print(req.url.replace(key, "YOUR_KEY") if key else req.url)

In [ ]:
# 🛟 Safety net: an ILLUSTRATIVE sample in the same shape as a real NeoWs answer.
# These are NOT real asteroids. It only exists so your feature can still show something if the API is down.
SAMPLE_NEOWS = {'element_count': 5,
 'near_earth_objects': {'2025-10-01': [{'id': '90001',
                                        'name': 'SAMPLE-1 (illustrative)',
                                        'is_potentially_hazardous_asteroid': False,
                                        'estimated_diameter': {'meters': {'estimated_diameter_min': 54.2,
                                                                          'estimated_diameter_max': 120.5}},
                                        'close_approach_data': [{'close_approach_date': '2025-10-01',
                                                                 'relative_velocity': {'kilometers_per_second': '12.3'},
                                                                 'miss_distance': {'kilometers': '5041879.1'}}]},
                                       {'id': '90002',
                                        'name': 'SAMPLE-2 (illustrative)',
                                        'is_potentially_hazardous_asteroid': False,
                                        'estimated_diameter': {'meters': {'estimated_diameter_min': 20.3,
                                                                          'estimated_diameter_max': 45.2}},
                                        'close_approach_data': [{'close_approach_date': '2025-10-01',
                                                                 'relative_velocity': {'kilometers_per_second': '8.7'},
                                                                 'miss_distance': {'kilometers': '11250430.8'}}]}],
                        '2025-10-02': [{'id': '90003',
                                        'name': 'SAMPLE-3 (illustrative)',
                                        'is_potentially_hazardous_asteroid': True,
                                        'estimated_diameter': {'meters': {'estimated_diameter_min': 139.5,
                                                                          'estimated_diameter_max': 310.0}},
                                        'close_approach_data': [{'close_approach_date': '2025-10-02',
                                                                 'relative_velocity': {'kilometers_per_second': '19.4'},
                                                                 'miss_distance': {'kilometers': '2310560.2'}}]},
                                       {'id': '90004',
                                        'name': 'SAMPLE-4 (illustrative)',
                                        'is_potentially_hazardous_asteroid': False,
                                        'estimated_diameter': {'meters': {'estimated_diameter_min': 10.3,
                                                                          'estimated_diameter_max': 22.8}},
                                        'close_approach_data': [{'close_approach_date': '2025-10-02',
                                                                 'relative_velocity': {'kilometers_per_second': '6.1'},
                                                                 'miss_distance': {'kilometers': '31870210.5'}}]},
                                       {'id': '90005',
                                        'name': 'SAMPLE-5 (illustrative)',
                                        'is_potentially_hazardous_asteroid': False,
                                        'estimated_diameter': {'meters': {'estimated_diameter_min': 39.7,
                                                                          'estimated_diameter_max': 88.3}},
                                        'close_approach_data': [{'close_approach_date': '2025-10-02',
                                                                 'relative_velocity': {'kilometers_per_second': '14.0'},
                                                                 'miss_distance': {'kilometers': '7640120.9'}}]}]}}

def sample_banner():
    print("⚠️  USING ILLUSTRATIVE SAMPLE DATA, not real NASA data. Fix the call to see real results.")

In [ ]:
# 🛟 Safety net: ILLUSTRATIVE earthquake sample in the shape of a USGS GeoJSON answer. NOT real earthquakes.
SAMPLE_QUAKES = {"features": [
    {"properties": {"mag": 5.8, "place": "SAMPLE region A (illustrative)", "time": 1759384800000}, "geometry": {"coordinates": [10.0, 20.0, 15.0]}},
    {"properties": {"mag": 4.7, "place": "SAMPLE region B (illustrative)", "time": 1759428000000}, "geometry": {"coordinates": [-30.0, 40.0, 10.0]}},
    {"properties": {"mag": 5.1, "place": "SAMPLE region C (illustrative)", "time": 1759298400000}, "geometry": {"coordinates": [100.0, -5.0, 30.0]}},
]}

In [ ]:
# 🎁 GIVEN: the asteroid table from the code-along, as one function (live call, sample as a last resort).
NEOWS_URL = "https://api.nasa.gov/neo/rest/v1/feed"

def neows_df(start, end):
    try:
        r = requests.get(NEOWS_URL, params={"start_date": start, "end_date": end, "api_key": API_KEY}, timeout=15)
        r.raise_for_status()
        feed = r.json()
        if not feed.get("near_earth_objects"):
            raise ValueError("Empty result")
    except (requests.RequestException, ValueError) as e:
        print("⚠️  NeoWs problem:", e)
        sample_banner()
        feed = SAMPLE_NEOWS
    rows = []
    for day, asteroids in feed["near_earth_objects"].items():
        for a in asteroids:
            ca = a["close_approach_data"][0]
            rows.append({
                "date": day, "name": a["name"],
                "diameter_m": float(a["estimated_diameter"]["meters"]["estimated_diameter_max"]),
                "speed_kms": float(ca["relative_velocity"]["kilometers_per_second"]),
                "miss_km": float(ca["miss_distance"]["kilometers"]),
                "hazardous": a["is_potentially_hazardous_asteroid"],
            })
    return pd.DataFrame(rows).sort_values("miss_km").reset_index(drop=True)

## 1 · Source A: asteroids for the last 7 days

### Your turn

In [ ]:
END = date.today()
START = ...                       # TODO: 6 days before END
asteroids = neows_df(..., ...)    # TODO: start and end as YYYY-MM-DD text
asteroids.head()

In [ ]:
check(lambda: len(asteroids) > 0 and "date" in asteroids.columns, "Source A is ready.", "You need a table of asteroids with a date column.")

## 2 · Source B: earthquakes from USGS
Endpoint: `https://earthquake.usgs.gov/fdsnws/event/1/query` · Parameters: `format=geojson`, `starttime`, `endtime` (YYYY-MM-DD), `minmagnitude` (use 4.5), `orderby=time`.

USGS answers a *GeoJSON*: look in `features`; each has `properties` (`mag`, `place`, `time`) and `geometry` → `coordinates` = **[longitude, latitude, depth]**.

### Your turn

In [ ]:
USGS_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"
usgs_params = {
    "format": ...,
    "starttime": ...,       # same days as your asteroids; endtime is exclusive, so add one day
    "endtime": ...,
    "minmagnitude": ...,
    "orderby": "time",
}
try:
    r = requests.get(USGS_URL, params=usgs_params, timeout=20)
    r.raise_for_status()
    quakes_raw = r.json()
    print("Status:", r.status_code, "| events:", len(quakes_raw["features"]))
except (requests.RequestException, ValueError) as e:
    print("⚠️  USGS problem:", e)
    # TODO: use SAMPLE_QUAKES and print the warning banner
    quakes_raw = ...

In [ ]:
check(lambda: "features" in quakes_raw, "Source B is ready.", "quakes_raw should be the GeoJSON dictionary with a 'features' list.")

## 3 · Pick the fields
For each earthquake keep `mag`, `place`, `time_ms` (the raw `time`), and `lon`, `lat`, `depth_km` from the coordinates. **Careful: longitude comes first.**

### Your turn

In [ ]:
rows = []
for f in quakes_raw["features"]:
    lon, lat, depth = ...          # TODO: unpack the coordinates (in this order!)
    rows.append({
        "mag": ...,
        "place": ...,
        "time_ms": ...,
        "lon": lon, "lat": lat, "depth_km": depth,
    })
quakes = pd.DataFrame(rows)
quakes.head()

In [ ]:
check(lambda: len(quakes) == len(quakes_raw["features"]) and quakes["lat"].abs().max() <= 90,
      "Coordinates in the right order (latitude is never above 90).", "If a latitude is above 90 you swapped longitude and latitude.")

## 4 · Make the join key: one date format on both sides
USGS time is **milliseconds since 1970 (UTC)**. Turn it into a plain date, then reduce to **one row per day**: the strongest earthquake of that day. Do the same type of date on the asteroid side.

### Your turn

In [ ]:
quakes["date"] = ...      # TODO: milliseconds -> datetime (unit="ms", utc=True) -> .dt.date
daily = ...               # TODO: one row per date with the strongest magnitude, columns: date, max_mag
asteroids["date"] = ...   # TODO: asteroid dates (text) -> .dt.date, so both sides have the same type
daily

In [ ]:
check(lambda: daily["date"].is_unique and type(daily["date"].iloc[0]) == type(asteroids["date"].iloc[0]),
      "One row per day, same date type on both sides.", "daily must have ONE row per date, and both date columns must be the same type.")

## 5 · Join!
Keep **every asteroid**, even on days with no earthquake (they get `NaN`). Then check that the join did not multiply rows.

### Your turn

In [ ]:
joined = asteroids.merge(..., on=..., how=...)     # TODO
print(len(asteroids), "->", len(joined), "rows")
joined.head()

In [ ]:
check(lambda: len(joined) == len(asteroids) and "max_mag" in joined.columns, "Joined, and no rows were multiplied.",
      "Use a left join on date, and make sure you joined `daily` (one row per day), not all earthquakes.")

## 6 · Familiar terms: how big is that energy?
A rough, back-of-the-envelope estimate (**order of magnitude only**):

- mass ≈ density × (4/3)π r³, with density ≈ 3,000 kg/m³ (rocky) and r = diameter / 2
- energy `E = ½ m v²` in joules (convert km/s → m/s)
- megatons of TNT = `E / 4.184e15`
- equivalent earthquake magnitude (Gutenberg–Richter): `M = (log10(E) − 4.8) / 1.5`

Assumptions to say out loud: the *maximum* estimated diameter, a fixed density, and **no** atmosphere effects.

### Your turn

In [ ]:
import math

def energy_joules(diameter_m, speed_kms, density=3000):
    ...      # TODO

def tnt_megatons(e_joules):
    ...      # TODO

def equivalent_magnitude(e_joules):
    ...      # TODO

joined["energy_mt"] = ...        # TODO: apply the functions row by row (hint: .apply(..., axis=1))
joined["equiv_quake_mag"] = ...
joined[["name", "diameter_m", "speed_kms", "energy_mt", "equiv_quake_mag", "max_mag"]].head()

In [ ]:
check(lambda: 1e16 < energy_joules(100, 12) < 1e18 and 5 < equivalent_magnitude(energy_joules(100, 12)) < 10,
      "Energy estimate is in a sensible range for a 100 m rock at 12 km/s.", "Check units: diameter in metres, speed converted from km/s to m/s.")

## 7 · The feature: one sentence a judge understands
For the asteroid with the **largest estimated energy**, print a sentence that compares it to an earthquake, and show the strongest real earthquake of the same period for context.

### Your turn

In [ ]:
top = ...      # TODO: the row with the largest energy_mt
print(f"{top['name']} ({top['diameter_m']:.0f} m wide, {top['speed_kms']:.1f} km/s):")
print(f"  rough energy ≈ {top['energy_mt']} megatons of TNT ≈ a magnitude {top['equiv_quake_mag']} earthquake")
# TODO: print the strongest earthquake in `quakes` (magnitude, place) for context

In [ ]:
check(lambda: top["energy_mt"] == joined["energy_mt"].max(), "Feature sentence built from your joined data.", "top should be the asteroid with the largest energy_mt.")

## 8 · Save a backup
Save the joined table so the demo works even if one API is down.

### Your turn

In [ ]:
os.makedirs("backup", exist_ok=True)
# TODO: joined -> backup/neows_usgs_joined.csv (no index)

In [ ]:
check(lambda: os.path.exists("backup/neows_usgs_joined.csv"), "Joined backup saved.", "Save the joined table as a CSV.")

## ✅ Done means
- [ ] two real calls (`200 OK`)
- [ ] one date format on both sides
- [ ] a left join that did not multiply rows
- [ ] one sentence feature in familiar terms
- [ ] you can **explain how the join works** (to your partner!)

**Think:** what other shared fields could join two NASA sources? Place (lat, lon)? An object ID?